#  Submission

## 1. Context: Final Submission Using Bimodal Model and Weighted Voting

In this notebook, we apply the **bimodal model** (based on both text and image inputs) using the **Weighted Voting** strategy during the model combination phase.

The objective is to generate final predictions on the **test set** provided in the Rakuten challenge.

The predicted class indices are then mapped back to their original **prdtypecode** values.  
Finally, the results are saved in **CSV format** to be submitted on the official challenge platform.



## 2. Import Required Libraries & Configuration

In [10]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
import sys
from pathlib import Path
import importlib

# Data science and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow and Keras libraries for model building and image preprocessing
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Scikit-learn metrics for model evaluation
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Setup dynamic project paths
CURRENT_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = CURRENT_DIR.parents[2]
sys.path.append(str(PROJECT_ROOT))

def get_relative_path(absolute_path):
    return str(Path(absolute_path).relative_to(PROJECT_ROOT))

print(f"Project Root Directory: {PROJECT_ROOT.name}")

# Load project config and modules from src
import config

Project Root Directory: Data_Scientist_Rakuten_Project-main


## 3. Load Submission Data

In [2]:
import src.data_preprocessing.data_loader
importlib.reload(src.data_preprocessing.data_loader)
from src.data_preprocessing.data_loader import load_submission_data

sub_data = load_submission_data(2)

[✔] Loaded `X_test_sub_cleaned_final` | Type: <class 'pandas.core.frame.DataFrame'>


,designation,description,text,productid,imageid,image_name
84916,folkmanis puppets marionnette theatre mini turtle,NaN,folkmanis puppets marionnette theatre mini turtle,516376098,1019294171,image_1019294171_product_516376098.jpg
84917,porte flamme gaxix flamebringer gaxix twilight...,NaN,porte flamme gaxix flamebringer twilight dragons,133389013,1274228667,image_1274228667_product_133389013.jpg


##  4. Load Pretrained Models

In [30]:
import src.model_inference.model_loader
importlib.reload(src.model_inference.model_loader)
from src.model_inference.model_loader import load_text_model,load_image_model

import src.model_inference.model_loader
importlib.reload(src.model_inference.model_loader)
from src.model_inference.model_loader import load_text_model,load_image_model

importlib.reload(config)

# Load best models dynamically from config
text_model_conv1d = load_text_model(config.BEST_TEXT_MODEL_CONV1D)
text_model_dnn = load_text_model(config.BEST_TEXT_MODEL_DNN)
image_model_xception = load_image_model(config.BEST_IMAGE_MODEL_XCEPTION)
image_model_inception = load_image_model(config.BEST_IMAGE_MODEL_INCEPTION)

print("\nText Conv1D model loaded:", text_model_conv1d is not None)
print("Text DNN model loaded:", text_model_dnn is not None)
print("Image Xception model loaded:", image_model_xception is not None)
print("Image Inception model loaded:", image_model_inception is not None)


# Regrouper dans un dictionnaire
models_dict= {
    'text_model_conv1d': text_model_conv1d,
    'text_model_dnn': text_model_dnn,
    'image_model_xception': image_model_xception,
    'image_model_inception': image_model_inception
}



print(f"\n[INFO] Models provided: {list(models_dict.keys())}")





[DEBUG] Base directory for text models: models\text\neural
[DEBUG] Checking if the model exists at: models\text\neural\conv1d_text_model_model_v2.h5
[INFO] Loading text model from models\text\neural\conv1d_text_model_model_v2.h5
[✔] Successfully loaded text model: conv1d_text_model_model_v2.h5

[DEBUG] Base directory for text models: models\text\neural
[DEBUG] Checking if the model exists at: models\text\neural\simple_DNN_text_model_model_V2.h5
[INFO] Loading text model from models\text\neural\simple_DNN_text_model_model_V2.h5
[✔] Successfully loaded text model: simple_DNN_text_model_model_V2.h5

[DEBUG] Base directory for image models: models\image\final_training
[DEBUG] Checking if the model exists at: models\image\final_training\image_model_xception_v1.hdf5
[DEBUG] Loading image model from : models\image\final_training\image_model_xception_v1.hdf5
[✔] Successfully loaded image  model: image_model_xception_v1.hdf5

[DEBUG] Base directory for image models: models\image\final_training

## 5. Model Prediction Using Voting Strategy

### 5.1 Combination 1: Simple DNN + Conv1D + Xception (Weighted Voting)

In [11]:
%%time 
import src.model_inference.predictions
importlib.reload(src.model_inference.predictions)
from src.model_inference.predictions import predict_combined_models


# Directory containing the test set images used for submission predictions
image_dir = Path(config.RAW_IMAGE_TEST_SUB_DIR)


# Select models for Combination 1
models_comb1 = {
    'text_model_dnn': models_dict['text_model_dnn'],
    'text_model_conv1d': models_dict['text_model_conv1d'],
    'image_model_xception': models_dict['image_model_xception']
}

# Define weights based on individual Weighted F1-scores (from validation phase)
weights_comb1 = [0.81, 0.80, 0.66]

# Generate predictions on the test set using Weighted Voting
combined_preds_comb1 = predict_combined_models(
    models=models_comb1,
    x_val_text=sub_data['text'],
    x_val_image=sub_data['image_name'],
    image_dir=image_dir,
    use_text=True,
    use_image=True,
    weights=weights_comb1
)

# Retrieve final predictions from the Weighted Voting output
submission_preds_comb1 = combined_preds_comb1['weighted_voting']

print(f"[INFO] Predictions completed for Combination 1 — Total samples: {len(submission_preds_comb1)}")


[INFO] Applying Max Voting (Hard Voting).
[INFO] Applying Max Voting (Soft Voting / Proba Average).
[INFO] Applying Weighted Average Voting.
[INFO] Applying Max Confidence Voting.
[INFO] Predictions completed for Combination 1 — Total samples: 13812
Wall time: 2min 19s


In [12]:
# Sanity check: ensure predictions were made
assert submission_preds_comb1 is not None, "No predictions found."
assert len(submission_preds_comb1) == len(sub_data), "Prediction count does not match submission data."

# Check unique predicted classes
unique_preds = np.unique(submission_preds_comb1)
print(f"[SANITY CHECK] Number of unique predicted classes: {len(unique_preds)}")
print(f"[SANITY CHECK] Predicted class labels: {unique_preds}")

# Check data type and shape
print(f"[SANITY CHECK] Predictions shape: {submission_preds_comb1.shape}")
print(f"[SANITY CHECK] First 10 predictions: {submission_preds_comb1[:10]}")


[SANITY CHECK] Number of unique predicted classes: 27
[SANITY CHECK] Predicted class labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26]
[SANITY CHECK] Predictions shape: (13812,)
[SANITY CHECK] First 10 predictions: [ 7 22 23 23  5  9 15 25 13 23]


### 5.2 Combination 2: Simple DNN + Conv1D + InceptionV3 (Weighted Voting)

In [13]:
%%time 
import src.model_inference.predictions
importlib.reload(src.model_inference.predictions)
from src.model_inference.predictions import predict_combined_models


# Directory containing the test set images used for submission predictions
image_dir = Path(config.RAW_IMAGE_TEST_SUB_DIR)


# Select models for Combination 2
models_comb2 = {
    'text_model_dnn': models_dict['text_model_dnn'],
    'text_model_conv1d': models_dict['text_model_conv1d'],
    'image_model_inception': models_dict['image_model_inception']
}

# Define weights based on individual Weighted F1-scores (from validation phase)
weights_comb2 = [0.81, 0.80, 0.60]

# Generate predictions on the test set using Weighted Voting
combined_preds_comb2 = predict_combined_models(
    models=models_comb2,
    x_val_text=sub_data['text'],
    x_val_image=sub_data['image_name'],
    image_dir=image_dir,
    use_text=True,
    use_image=True,
    weights=weights_comb2
)

# Retrieve final predictions from the Weighted Voting output
submission_preds_comb2 = combined_preds_comb2['weighted_voting']

print(f"[INFO] Predictions completed for Combination 2 — Total samples: {len(submission_preds_comb2)}")


[INFO] Applying Max Voting (Hard Voting).
[INFO] Applying Max Voting (Soft Voting / Proba Average).
[INFO] Applying Weighted Average Voting.
[INFO] Applying Max Confidence Voting.
[INFO] Predictions completed for Combination 2 — Total samples: 13812
Wall time: 2min


In [14]:
# Sanity check for Combination 2 predictions
assert submission_preds_comb2 is not None, "No predictions found for Combination 2."
assert len(submission_preds_comb2) == len(sub_data), "Prediction count does not match submission data."

# Check unique predicted classes
unique_preds = np.unique(submission_preds_comb2)
print(f"[SANITY CHECK] Number of unique predicted classes: {len(unique_preds)}")
print(f"[SANITY CHECK] Predicted class labels: {unique_preds}")

# Check data type and shape
print(f"[SANITY CHECK] Predictions shape: {submission_preds_comb2.shape}")
print(f"[SANITY CHECK] First 10 predictions: {submission_preds_comb2[:10]}")


[SANITY CHECK] Number of unique predicted classes: 27
[SANITY CHECK] Predicted class labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26]
[SANITY CHECK] Predictions shape: (13812,)
[SANITY CHECK] First 10 predictions: [ 7  5 23 23 16  9 15 25 13 23]


## 6. Map Predicted Labels to prdtypecode

In [15]:
import src.data_preprocessing.data_loader
importlib.reload(src.data_preprocessing.data_loader)
from src.data_preprocessing.data_loader import load_product_code_mapping

# Charger et afficher le mapping
mapping_df = load_product_code_mapping()

# Afficher les premières lignes du DataFrame du mapping
print("Product Code Mapping:")
display(mapping_df.head())

[✔] Loaded `prdtypecode_mapping` | Type: <class 'pandas.core.frame.DataFrame'>
Product Code Mapping:


,Original prdtypecode,Encoded target,Label
0,10,0,Adult Books
1,40,1,Imported Video Games
2,50,2,Video Games Accessories
3,60,3,Games and Consoles
4,1140,4,Figurines and Toy Pop


In [20]:
import src.data_preprocessing.data_loader  
importlib.reload(src.data_preprocessing.data_loader )
from src.data_preprocessing.data_loader import load_product_code_mapping, map_encoded_predictions_to_labels


mapped_df_com1 = map_encoded_predictions_to_labels(
    predictions=combined_preds_comb1['weighted_voting'],  # ou 'hard_voting' ou 'soft_voting'
    mapping_df=mapping_df
)

display(mapped_df_com1)

[DEBUG] Unique predictions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]
[DEBUG]  Encoded target values available in mapping table: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26]


,Predicted Encoded,prdtypecode,Label
0,7,1280,Toys for Children
1,22,2582,"Furniture, Kitchen, and Garden"
2,23,2583,Piscine and Spa
3,23,2583,Piscine and Spa
4,5,1160,Playing Cards
...,...,...,...
13807,13,1560,Interior Furniture and Bedding
13808,19,2403,Children Books and Magazines
13809,23,2583,Piscine and Spa
13810,22,2582,"Furniture, Kitchen, and Garden"


## 7. Export Predictions to CSV for Submission

### 7.1 Generate and Export — Combination 1: Simple DNN + Conv1D + Xception (Weighted Voting)

In [23]:
import src.export_utils
importlib.reload( src.export_utils)
from src.export_utils import generate_submission_file
importlib.reload(config)
from datetime import datetime

# Get current date in YYYYMMDD format
today_str = datetime.now().strftime("%Y%m%d")

# Build filename with date
csv_sub_file_name_comb1 = f"sub_dnn_conv1d_xcep_weighted_{today_str}.csv"

# Generate the final CSV submission file for Combination 1
# This will include only the 'prdtypecode' column, aligned with the test set (sub_data),
# and save it to the specified submission directory
generate_submission_file(
    mapped_preds_df=mapped_df,
    sub_data=sub_data,
    output_name=csv_sub_file_name_comb1,
    sub_dir=config.SUBMISSION_DIR
)

print(f"[INFO] Submission file generated: {config.SUBMISSION_DIR}/{csv_sub_file_name_comb1}")

[INFO] Generating submission file: sub_dnn_conv1d_xcep_weighted_20250510.csv
[✔] Submission file saved at: submissions\sub_dnn_conv1d_xcep_weighted_20250510.csv
[INFO] Submission file generated: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\submissions/sub_dnn_conv1d_xcep_weighted_20250510.csv


### 7.2 Generate and Export — Combination 2: Simple DNN + Conv1D + Inception (Weighted Voting)

In [26]:
import src.export_utils
importlib.reload( src.export_utils)
from src.export_utils import generate_submission_file
importlib.reload(config)
from datetime import datetime

# Get current date in YYYYMMDD format
today_str = datetime.now().strftime("%Y%m%d")

# Build filename with date
csv_sub_file_name_comb2 = f"sub_dnn_conv1d_incep_weighted_{today_str}.csv"

# Generate the final CSV submission file for Combination 2
# This will include only the 'prdtypecode' column, aligned with the test set (sub_data),
# and save it to the specified submission directory
generate_submission_file(
    mapped_preds_df=mapped_df,
    sub_data=sub_data,
    output_name=csv_sub_file_name_comb2,
    sub_dir=config.SUBMISSION_DIR
)

print(f"[INFO] Submission file generated: {config.SUBMISSION_DIR}/{csv_sub_file_name_comb2}")

[INFO] Generating submission file: sub_dnn_conv1d_incep_weighted_20250510.csv
[✔] Submission file saved at: submissions\sub_dnn_conv1d_incep_weighted_20250510.csv
[INFO] Submission file generated: D:\Data_Science\Append_Data_Engineer_AWS_MLOPS\Data_Scientist_Rakuten_Project-main\submissions/sub_dnn_conv1d_incep_weighted_20250510.csv
